In [8]:
import sys
import os
import json
import time
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import wfdb
from scipy.signal import butter, filtfilt, iirnotch, resample_poly

# ------------------------------------------------------------
# 将项目根目录加入 python 路径（确保可以导入 src 中的模块）
# 如果你的 notebook 放在 notebooks/ 下，则用下面这行
# ------------------------------------------------------------
project_root = os.path.abspath('../')   # 根据实际位置调整
if project_root not in sys.path:
    sys.path.append(project_root)

# ------------------------------------------------------------
# 设置数据路径（根据你的实际情况修改）
# ------------------------------------------------------------
RAW_DATA_DIR = os.path.join(project_root, 'data', 'raw', 'ptb-xl_1.0.2')
CSV_PATH      = os.path.join(RAW_DATA_DIR, 'ptbxl_database.csv')
OUTPUT_DIR    = os.path.join(project_root, 'data', 'processed')

print("✅ 路径设置完成")
print("原始数据目录:", RAW_DATA_DIR)
print("预处理输出目录:", OUTPUT_DIR)

✅ 路径设置完成
原始数据目录: d:\Code (VS Code)\EGC_Model\data\raw\ptb-xl_1.0.2
预处理输出目录: d:\Code (VS Code)\EGC_Model\data\processed


In [9]:
from typing import cast
import numpy as np
from scipy.signal import butter, filtfilt, iirnotch, resample_poly

def bandpass_filter(data, lowcut=0.5, highcut=45.0, fs=500, order=4):
    """零相位带通滤波（Butterworth）"""
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    # cast 强制声明返回类型，消除 Pylance 误报
    b, a = cast(
        tuple[np.ndarray, np.ndarray],
        butter(order, [low, high], btype='band', output='ba')
    )
    return filtfilt(b, a, data, axis=-1)

def notch_filter(data, freq=50.0, fs=500, quality=30):
    """陷波滤波器（去除工频干扰）"""
    nyq = 0.5 * fs
    b, a = cast(
        tuple[np.ndarray, np.ndarray],
        iirnotch(freq, quality, fs)
    )
    return filtfilt(b, a, data, axis=-1)

def resample(data, orig_fs, target_fs=100):
    """重采样到目标频率"""
    return resample_poly(data, up=10, down=int(10 / (orig_fs / target_fs)))

def zscore_normalize(data):
    """按导联进行 Z-score 标准化"""
    mean = data.mean(axis=-1, keepdims=True)
    std = data.std(axis=-1, keepdims=True) + 1e-8
    return (data - mean) / std

print("✅ 信号处理函数定义完毕")

✅ 信号处理函数定义完毕


In [10]:
# 加载 CSV
df = pd.read_csv(CSV_PATH)
print(f"原始记录数: {len(df)}")

# 根据上一步探索结果清洗年龄异常值（保留 0‑120 岁）
df = df[(df['age'] >= 0) & (df['age'] <= 120)].copy()
print(f"年龄清洗后记录数: {len(df)}")

# [可选] 丢弃探索中发现缺失率极高的列，例如 height, weight, electrodes_problems 等
# 如果这些列几乎全为空，可以提前删除以减少干扰
cols_to_drop = ['height', 'weight', 'electrodes_problems', 'static_noise', 'burst_noise',
                'electrode_problems', 'extra_beats', 'pacemaker_note']
for col in cols_to_drop:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

print(f"丢弃高缺失列后，列数: {df.shape[1]}")

原始记录数: 21801
年龄清洗后记录数: 21508
丢弃高缺失列后，列数: 38


In [11]:
# 按照官方建议的 fold 划分
train_folds = list(range(1, 9))   # fold 1~8
val_fold = 9
test_fold = 10

train_df = df[df['strat_fold'].isin(train_folds)].copy()
val_df   = df[df['strat_fold'] == val_fold].copy()
test_df  = df[df['strat_fold'] == test_fold].copy()

print(f"训练集: {len(train_df)} 条")
print(f"验证集: {len(val_df)} 条")
print(f"测试集: {len(test_df)} 条")

训练集: 17203 条
验证集: 2141 条
测试集: 2164 条


In [12]:
import wfdb

def process_single_record(file_path, record_fs=500, target_fs=100, duration=10):
    """
    处理一条 ECG 记录：
    1. 读取 WFDB 格式的 12 导联信号
    2. 带通滤波 -> 陷波 -> 重采样 -> 截取/补齐 -> Z-score 归一化
    返回: (12, target_length) 的 numpy array
    """
    # 读取信号
    record = wfdb.rdrecord(file_path)

    # 类型收窄：确保是单记录 Record（而非 MultiRecord），同时消除 Pylance 警告
    if not isinstance(record, wfdb.Record):
        raise TypeError(
            f"期望单个 Record，但 rdrecord 返回了 {type(record).__name__}。"
            f"文件: {file_path}"
        )

    # 确保关键字段非空
    if record.p_signal is None:
        raise ValueError(f"记录 {file_path} 中没有信号数据 (p_signal 为 None)")
    if record.fs is None:
        raise ValueError(f"记录 {file_path} 中没有采样率信息 (fs 为 None)")

    signals = record.p_signal.T  # shape (12, N)
    orig_fs = int(record.fs)     # 显式转为 int，消除类型歧义

    # 1. 带通滤波
    signals = bandpass_filter(signals, fs=orig_fs)
    # 2. 陷波
    signals = notch_filter(signals, fs=orig_fs)
    # 3. 重采样
    signals = resample(signals, orig_fs=orig_fs, target_fs=target_fs)
    # 4. 固定长度截取/补齐
    target_len = target_fs * duration
    if signals.shape[1] >= target_len:
        signals = signals[:, :target_len]
    else:
        pad_width = target_len - signals.shape[1]
        signals = np.pad(signals, ((0, 0), (0, pad_width)), mode='constant')
    # 5. Z-score 归一化
    signals = zscore_normalize(signals)

    return signals.astype(np.float32)

In [13]:
# 处理参数
TARGET_FS = 100
DURATION  = 10

# 遍历划分
splits = {
    'train': train_df,
    'val': val_df,
    'test': test_df
}

for split_name, split_df in splits.items():
    print(f"\n===== 处理 {split_name} 集 =====")
    split_dir = os.path.join(OUTPUT_DIR, split_name)
    os.makedirs(split_dir, exist_ok=True)

    signals_list = []
    labels_list  = []

    # 遍历每一条记录
    for _, row in tqdm(split_df.iterrows(), total=len(split_df)):
        # 构建记录路径（不带后缀）
        rel_path = row['filename_lr']   # e.g. 'records100/00000/00001_lr'
        full_path = os.path.join(RAW_DATA_DIR, rel_path)

        # 如果文件不存在则跳过（可能数据不完整）
        if not os.path.exists(full_path + '.dat'):
            print(f"警告: 找不到文件 {full_path}.dat，跳过该记录 (ecg_id={row['ecg_id']})")
            continue

        try:
            # 处理信号
            signal = process_single_record(full_path,
                                           record_fs=500,
                                           target_fs=TARGET_FS,
                                           duration=DURATION)
        except Exception as e:
            print(f"错误: 处理 ecg_id={row['ecg_id']} 时出错: {e}")
            continue

        signals_list.append(signal)

        # ✅ 保存原始标签列 —— 只选 CSV 中实际存在的列
        label_cols = ['scp_codes', 'strat_fold']
        for col in ['diagnostic_class', 'report', 'validated_by', 'second_opinion']:
            if col in row.index:
                label_cols.append(col)
        labels_list.append(row[label_cols])

    # 转换为数组并保存
    if len(signals_list) == 0:
        print(f"⚠️ {split_name} 集无有效数据，跳过保存")
        continue

    signals_array = np.stack(signals_list)   # (N, 12, 1000)
    np.save(os.path.join(split_dir, 'signals.npy'), signals_array)

    labels_df = pd.DataFrame(labels_list)
    labels_df.to_csv(os.path.join(split_dir, 'labels.csv'), index=False)

    print(f"✅ {split_name}: 保存 {signals_array.shape[0]} 条信号，形状 {signals_array.shape}")


===== 处理 train 集 =====


  0%|          | 0/17203 [00:00<?, ?it/s]

警告: 找不到文件 d:\Code (VS Code)\EGC_Model\data\raw\ptb-xl_1.0.2\records100/00000/00490_lr.dat，跳过该记录 (ecg_id=490)
错误: 处理 ecg_id=990 时出错: [Errno 2] No such file or directory: 'd:/Code (VS Code)/EGC_Model/data/raw/ptb-xl_1.0.2/records100/00000/00990_lr.hea'
警告: 找不到文件 d:\Code (VS Code)\EGC_Model\data\raw\ptb-xl_1.0.2\records100/01000/01991_lr.dat，跳过该记录 (ecg_id=1991)
错误: 处理 ecg_id=2993 时出错: [Errno 2] No such file or directory: 'd:/Code (VS Code)/EGC_Model/data/raw/ptb-xl_1.0.2/records100/02000/02993_lr.hea'
错误: 处理 ecg_id=6500 时出错: [Errno 2] No such file or directory: 'd:/Code (VS Code)/EGC_Model/data/raw/ptb-xl_1.0.2/records100/06000/06500_lr.hea'
警告: 找不到文件 d:\Code (VS Code)\EGC_Model\data\raw\ptb-xl_1.0.2\records100/07000/07501_lr.dat，跳过该记录 (ecg_id=7501)
错误: 处理 ecg_id=8004 时出错: [Errno 2] No such file or directory: 'd:/Code (VS Code)/EGC_Model/data/raw/ptb-xl_1.0.2/records100/08000/08004_lr.hea'
警告: 找不到文件 d:\Code (VS Code)\EGC_Model\data\raw\ptb-xl_1.0.2\records100/09000/09005_lr.dat，跳过该记录 (ecg

  0%|          | 0/2141 [00:00<?, ?it/s]

警告: 找不到文件 d:\Code (VS Code)\EGC_Model\data\raw\ptb-xl_1.0.2\records100/04000/04498_lr.dat，跳过该记录 (ecg_id=4498)
错误: 处理 ecg_id=4998 时出错: [Errno 2] No such file or directory: 'd:/Code (VS Code)/EGC_Model/data/raw/ptb-xl_1.0.2/records100/04000/04998_lr.hea'
错误: 处理 ecg_id=10509 时出错: [Errno 2] No such file or directory: 'd:/Code (VS Code)/EGC_Model/data/raw/ptb-xl_1.0.2/records100/10000/10509_lr.hea'
警告: 找不到文件 d:\Code (VS Code)\EGC_Model\data\raw\ptb-xl_1.0.2\records100/16000/16524_lr.dat，跳过该记录 (ecg_id=16524)
错误: 处理 ecg_id=18526 时出错: [Errno 2] No such file or directory: 'd:/Code (VS Code)/EGC_Model/data/raw/ptb-xl_1.0.2/records100/18000/18526_lr.hea'
✅ val: 保存 2136 条信号，形状 (2136, 12, 1000)

===== 处理 test 集 =====


  0%|          | 0/2164 [00:00<?, ?it/s]

警告: 找不到文件 d:\Code (VS Code)\EGC_Model\data\raw\ptb-xl_1.0.2\records100/06000/06000_lr.dat，跳过该记录 (ecg_id=6000)
警告: 找不到文件 d:\Code (VS Code)\EGC_Model\data\raw\ptb-xl_1.0.2\records100/11000/11010_lr.dat，跳过该记录 (ecg_id=11010)
错误: 处理 ecg_id=14021 时出错: [Errno 2] No such file or directory: 'd:/Code (VS Code)/EGC_Model/data/raw/ptb-xl_1.0.2/records100/14000/14021_lr.hea'
✅ test: 保存 2161 条信号，形状 (2161, 12, 1000)


In [14]:
metadata = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'source_csv': CSV_PATH,
    'target_fs': TARGET_FS,
    'duration_sec': DURATION,
    'filtering': {
        'bandpass_lowcut': 0.5,
        'bandpass_highcut': 45.0,
        'notch_freq': 50.0,
        'normalization': 'zscore_per_lead'
    },
    'splits': {
        'train_count': len(train_df),
        'val_count': len(val_df),
        'test_count': len(test_df)
    },
    'age_cleaning': 'keep only 0-120',
    'dropped_columns': cols_to_drop
}

# 保存到 processed 根目录
metadata_path = os.path.join(OUTPUT_DIR, 'metadata.json')
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("✅ 元数据已保存至:", metadata_path)
print(json.dumps(metadata, indent=2, ensure_ascii=False))

✅ 元数据已保存至: d:\Code (VS Code)\EGC_Model\data\processed\metadata.json
{
  "timestamp": "2026-06-03 23:19:36",
  "source_csv": "d:\\Code (VS Code)\\EGC_Model\\data\\raw\\ptb-xl_1.0.2\\ptbxl_database.csv",
  "target_fs": 100,
  "duration_sec": 10,
  "filtering": {
    "bandpass_lowcut": 0.5,
    "bandpass_highcut": 45.0,
    "notch_freq": 50.0,
    "normalization": "zscore_per_lead"
  },
  "splits": {
    "train_count": 17203,
    "val_count": 2141,
    "test_count": 2164
  },
  "age_cleaning": "keep only 0-120",
  "dropped_columns": [
    "height",
    "weight",
    "electrodes_problems",
    "static_noise",
    "burst_noise",
    "electrode_problems",
    "extra_beats",
    "pacemaker_note"
  ]
}


In [15]:
# 快速验证生成的文件是否正确
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(OUTPUT_DIR, split)
    if not os.path.exists(split_dir):
        continue
    signals = np.load(os.path.join(split_dir, 'signals.npy'))
    labels = pd.read_csv(os.path.join(split_dir, 'labels.csv'))
    print(f"{split}: 信号形状 {signals.shape}, 标签列 {list(labels.columns)}")

train: 信号形状 (17181, 12, 1000), 标签列 ['scp_codes', 'strat_fold', 'report', 'validated_by', 'second_opinion']
val: 信号形状 (2136, 12, 1000), 标签列 ['scp_codes', 'strat_fold', 'report', 'validated_by', 'second_opinion']
test: 信号形状 (2161, 12, 1000), 标签列 ['scp_codes', 'strat_fold', 'report', 'validated_by', 'second_opinion']
